In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as broadcast

transactions = spark.read.format("delta").load("/delta/bronze/transactions")
customers = spark.read.format("delta").load("/delta/bronze/customers")

# Transaction tables has more than 10M rows, so we can filter out unwanted data from the table to optimize and reduce shuffle.
transactions = transactions.filter(F.col("amount") > 0)

# If the primary key of the table is NULL, then we don't need those data as well.
transactions = transactions.filter(F.col("transaction_id").isNotNull())

# This is coming from client database, so we are not sure how the data is! - so do the basic cleaning
customers = customers.filter(F.col("customer_id").isNotNull())

"""We Need to customer details for the transactions, so we are joining the tables"""
"""Need to Caution in the table size, as this will cause shuffle and spikes the cost"""

"""
Check both the table size or row
           * In our case Transactions table has above 10M records, so the table size will be around 10GB~ .
           * Customer table has 10k records, so the table size will be around 100MB.
    
So in this case we can use Broadcast Join, since it is safe and avoids Shuffling.
"""
# Why?
# Kafka retries can create duplicate messages, Banking system cannot process duplicate transaction
transactions = transactions.withWatermark("transaction_time","10 minutes").dropDuplicates(['transaction_id'])

# Banks must know when the record is loaded and what is the source system
transactions_with_customers = transactions.join(broadcast(customers), on= "customer_id", how="left")

transaction_with_customers = (
    transactions_with_customers.withColumn(
        "record_inserted_date",
        current_timestamp()
    ).withColumn(
        "source_system",
        lit("KAFKA")
    )
)


In [0]:
path = "/delta/silver/transactions_with_customers"

if DeltaTable.isDeltaTable(spark, path):
    delta = DeltaTable.path(spark, path)

    delta.alias("d").merge(
        transactions_with_customers.alias("tc"),
        "tc.transaction_id = d.transaction_id"
    ).whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

else:
    transactions_with_customers.write.format("delta").save(path)